# 🏛️ oracc-parser — Quickstart

This notebook demonstrates how to get started using the oracc-parser package, focusing on the main functionalities and relevant data stored for users on Zenodo.

## 🎯 Goals of this notebook

1. **Get the catalogue data** — download catalogues (metadata) from Zenodo
2. **Parse a project** — load word CSVs, inspect the project catalogue, and explore individual tablets
3. **Get flat DataFrames** — metadata, transliterations, and full combined tables ready for machine learning analysis
4. **Export** — save results as CSV or JSONL
5. **Download all available projects** — takes 20-25 minutes to download to disc

## Prerequisite: install the package

In [1]:
# Install oracc-parser from PyPI
!pip install oracc-parser -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.3/140.3 kB 1.4 MB/s eta 0:00:00


## Prerequisite: set up the data directory

A DATA_DIR must be defined for the package to work correctly. This is where the data is stored after download from Zenodo or the live ORACC server.

**Option 1**: Colab

Set it up as a folder in your Google Drive (requires mounting Google Drive).

**Option 2**: Run Locally

Set it up anywhere you want on your computer.

Choose whichever option below you prefer (comment out the other one).

In [2]:
# --- Option 1: persist downloaded data across Colab sessions using Google Drive ---
# Without this, data downloads to /content/oracc_data and is lost when the session ends.
# Uncomment the lines below to mount your Drive and store data there persistently.
#
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["ORACC_DATA_DIR"] = "/content/drive/MyDrive/oracc_data"

# --- Option 2: run locally (not in Colab) ---
# If you are running this notebook locally, set the data directory here.
# Uncomment the lines below and set the path to where you want data stored.
#
# import oracc_parser.settings as settings
# from pathlib import Path
# settings.DATA_DIR = Path("/path/to/your/oracc_data")


Mounted at /content/drive


## 1. Get the catalogue data (Zenodo)

Download project catalogues from Zenodo (doi: 10.5281/zenodo.20625379). These contain all the metadata stored per text by project.

> ❗ Metadata is important to have when assessing results of any machine learning model!

In [3]:
from oracc_parser.settings import CATALOGUE_DIR
from oracc_parser.download.fetch_data import fetch_data

# Check if catalogues were already downloaded
catalogues_ok = CATALOGUE_DIR.exists() and any(CATALOGUE_DIR.glob("*.csv"))

if not catalogues_ok:
    print("Downloading catalogues from Zenodo...")
    fetch_data()
else:
    print(f"Catalogues found in {CATALOGUE_DIR} ({len(list(CATALOGUE_DIR.glob('*.csv')))} projects)")

Catalogues found in /content/drive/MyDrive/oracc_data/catalogues (138 projects)


## 2. Parse a project

### How project data is organized on Zenodo

Most of the pre-processing work has already been done on the majority of ORACC projects.

> See notebook04 for an in-depth explanation on what has been done and how to process a new ORACC project from scratch!

The pre-processed word-level CSVs (one word per CSV row) are stored on the Zenodo repository accompanying this package.

ORACC projects follow a two-level hierarchy: an **umbrella project** (e.g. `saao`) groups related **sub-projects** (e.g. `saao/saa01`, `saao/saa02`, …). On Zenodo, word CSVs are stored as one zip per umbrella — so `saao.zip` contains the data for all 22 State Archives of Assyria sub-projects.

When you request a sub-project for the first time, the package downloads the entire umbrella zip, extracts all sub-projects it contains, and caches them locally. Subsequent requests for any project in the same umbrella (e.g. switching from `saao/saa01` to `saao/saa02`) are instant — no further download needed.

Some projects have no umbrella (e.g. `babcity`, `balt`, and many more) and are stored as their own single-project zip.

For demonstration purposes, `babcity` was picked below as it is a small project with fast download.

**Change the variable `PROJECT` to any valid oracc project or subproject name to test with different projects!**

> ❗ Note: When downloading subprojects, you need to provide the full name, e.g. `saao/saa01`, NOT `saa01`.

In [4]:
from oracc_parser import parse_project, RunConfig, get_metadata_table

PROJECT = "babcity"
LIMIT = 10  # Set to None to load all tablets

config = RunConfig(fetch_translations=False, limit=LIMIT)
records = parse_project(PROJECT, config=config)
print(f"Loaded {len(records)} tablets from {PROJECT}")

get_metadata_table(records)

17:45:03 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
Processing babcity: 100%|██████████| 10/10 [00:00<00:00, 25.77it/s]
17:45:04 | INFO    | oracc_parser | Processed 10 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 10 tablets for babcity from word CSVs


Loaded 10 tablets from babcity


,id,project,text_id,genre,archive,provenance,pleiades_id,state_supergroup,period,start_year,end_year,accession_museum_publication_numbers,secondary_literature,credits,cite_as
0,babcity_P261352,babcity,P261352,Legal,Murašû Archive,Nippur,912910,Mix,achaemenid,-547,-331,"BE 10, 056; CBS 05160",Clay Aram.Endors. 306 #17(Aram.only); PBS 02/1...,Based on a transliteration by Heather D. Baker...,Please cite this page as http://oracc.org/babc...
1,babcity_P285916,babcity,P285916,Legal,Egibi Archive,Babylon,893951,Mix,achaemenid,-547,-331,"Rental 130; Sp 2, 0016; BM 034544","Zawadzki, Rental of Houses 130","Based on the edition of Stefan Zawadzki, The R...",Please cite this page as http://oracc.org/babc...
2,babcity_P295135,babcity,P295135,Legal,Ea-ilutu-bani Archive,Borsippa,893964,Mix,achaemenid,-547,-331,BRM 1 68; MLC 330,"Joannes,Archives de Borsippa 94 (Tr) and 325 (T)","Adapted, lemmatised and translated by Lindsey ...",Please cite this page as http://oracc.org/babc...
3,babcity_P295289,babcity,P295289,Legal,Ea-ilutu-bani Archive,Borsippa,893964,Mix,Neo-Babylonian,-626,-539,BRM 1 58; MLC 00491,"Joannes, Archives de Borsippa 113 (Tr) and 323(T)","Adapted, lemmatised and translated by Lindsey ...",Please cite this page as http://oracc.org/babc...
4,babcity_P296350,babcity,P296350,Legal,Ea-ilutu-bani Archive,Borsippa,893964,Mix,Neo-Babylonian,-626,-539,BRM 1 43; MLC 01668,,"Adapted, lemmatised and translated by Lindsey ...",Please cite this page as http://oracc.org/babc...
5,babcity_P296437,babcity,P296437,Legal,Egibi Archive,Babylon,893951,Mix,achaemenid,-547,-331,BRM 1 74; MLC 1767,,Based on a transliteration by Heather D. Baker...,Please cite this page as http://oracc.org/babc...
6,babcity_P297595,babcity,P297595,Legal,Mušēzib-Marduk Archive,Uruk,912986,Mix,Neo-Assyrian,-911,-612,BaAr 5 21; NBC 04576,"Beaulieu, CBCY 1, p. 29","Based on the edition of Grant Frame, The Archi...",Please cite this page as http://oracc.org/babc...
7,babcity_P299751,babcity,P299751,Legal,Mušēzib-Marduk Archive,Uruk,912986,Mix,Neo-Assyrian,-911,-612,BaAr 5 25; NBC 08392,"Brinkman and Kennedy 1983: 40, L.4; Ellis 1984...","Based on the edition of Grant Frame, The Archi...",Please cite this page as http://oracc.org/babc...
8,babcity_P299753,babcity,P299753,Legal,Mušēzib-Marduk Archive,Uruk,912986,Mix,Neo-Assyrian,-911,-612,BaAr 5 26; NBC 08393,"Brinkman and Kennedy 1983: 45, L.94; Ellis 198...","Based on the edition of Grant Frame, The Archi...",Please cite this page as http://oracc.org/babc...
9,babcity_P311584,babcity,P311584,Legal,Mušēzib-Marduk Archive,Uruk,912986,Mix,Neo-Assyrian,-911,-612,BaAr 5 16; YBC 11413,Goetze 1944: 44 n. 13; Brinkman and Kennedy 19...,"Based on the edition of Grant Frame, The Archi...",Please cite this page as http://oracc.org/babc...


### View one text

The texts after processing are stored as a `TabletRecord`, a structured Python object (Pydantic model) with typed fields for metadata  (provenance, period, genre, etc.) and content (transliteration, normalization, translation, etc.).

For viewing the texts, you can use `record_to_word_dataframe` for a word-level csv (same format in which the documents are stored on Zenodo), or as string representations of the full text using by accessing the `content` dictionary inside the `TabletRecord`.

In [5]:
from oracc_parser.io.word_csv import record_to_word_dataframe

tablet = records[0] # using the first TabletRecord as an example

word_df = record_to_word_dataframe(records[0])
print(f"Word-level CSV for {records[0].metadata.id_text}: {len(word_df)} rows")
display(word_df.head(10))
print("------------------------------")

c = tablet.content # access the full string representation of the text as a transliteration, lemmatization, normalization, or Unicode cuneiform

print("TRANSLITERATION:")
print(f"{c.transliterated_str_representation.text[:100] if c.transliterated_str_representation else '(none)'}")
print("------------------------------")
print("LEMMATIZATION:")
print(f"{c.lemmatized_str_representation.text[:100] if c.lemmatized_str_representation else '(none)'}")
print("------------------------------")
print("NORMALIZATION:")
print(f"{c.normalized_str_representation.text[:100] if c.normalized_str_representation else '(none)'}")
print("------------------------------")
print("UNICODE CUNEIFORM:")
print(f"{c.unicode_str_representation.text[:100] if c.unicode_str_representation else '(none)'}")

Word-level CSV for P261352: 88 rows


,text_id,project,word_index,frag,ref,inst,form,lemma_form,sense,norm,raw_pos,lang,line,signs_reading,signs_break_pct,unicode,break_info
0,P261352,babcity,0,1,P261352.3.1,n,1,,,,n,akk-x-neobab,3,[1],1.00,𒁹,missing
1,P261352,babcity,1,1/2,P261352.3.2,n,1/2,,,,n,akk-x-neobab,3,[1/2],1.00,𒈦,missing
2,P261352,babcity,2,MA.NA,P261352.3.3,manû[unit]N,MA.NA,manû,unit,manû,N,akk-x-neobab,3,[MA].[NA],1.00,𒈠; 𒈾,missing; missing
3,P261352,babcity,3,KU₃.BABBAR,P261352.3.4,kaspi[silver]N,KU₃.BABBAR,kaspu,silver,kaspi,N,akk-x-neobab,3,[KU₃].[BABBAR],1.00,𒆬; 𒌓,missing; missing
4,P261352,babcity,4,i-di,P261352.3.5,+idu[arm//rent]N'N$idī,i-di,idu,rent,idī,N,akk-x-neobab,3,[i-di],1.00,𒄿; 𒁲,missing; missing
5,P261352,babcity,5,E₂,P261352.3.6,bīti[house]N,E₂,bītu,house,bīti,N,akk-x-neobab,3,[E₂],1.00,𒂍,missing
6,P261352,babcity,6,ša₂,P261352.3.7,ša[of]DET,ša₂,ša,of,ša,DET,akk-x-neobab,3,[ša₂],1.00,𒃻,missing
7,P261352,babcity,7,MU,P261352.3.8,šatti[year]N,MU,šattu,year,šatti,N,akk-x-neobab,3,[MU],1.00,𒈬,missing
8,P261352,babcity,8,1-KAM],P261352.3.9,n,1-KAM,,,,n,akk-x-neobab,3,[1-KAM],1.00,𒁹; 𒄰,missing; missing
9,P261352,babcity,9,{m}da-ri-ia-⸢a⸣-muš,P261352.4.1,Darius[1]RN,{m}da-ri-ia-a-muš,Darius,1,Darius,RN,akk-x-neobab,4,{m}da-ri-ia-⸢a⸣-muš,0.08,𒁹; 𒁕; 𒊑; 𒅀; 𒀀; 𒈲,complete; complete; complete; complete; damage...


------------------------------
TRANSLITERATION:
1 1/2 MA.NA KU₃.BABBAR i-di E₂ ša₂ MU 1-KAM]
{m}da-ri-ia-⸢a⸣-muš LUGAL ša₂ {[m}{d}EN-it-tan-nu]
A ša
------------------------------
LEMMATIZATION:
UNK UNK manû kaspu idu bītu ša šattu UNK
Darius šarru ša Bel-ittannu
māru ša Enlil-uballiṭ ša ina pā
------------------------------
NORMALIZATION:
UNK UNK manû kaspi idī bīti ša šatti UNK
Darius šarri ša Bel-ittannu
māru ša Enlil-uballiṭ ša ina pā
------------------------------
UNICODE CUNEIFORM:
𒁹 𒈦 𒈠𒈾 𒆬𒌓 𒄿𒁲 𒂍 𒃻 𒈬 𒁹𒄰
𒁹𒁕𒊑𒅀𒀀𒈲 𒈗 𒃻 𒁹𒀭𒂗𒀉𒆗𒉡
𒀀 𒃻 𒁹𒀭𒂗𒆤𒁷𒀉 𒃻 𒀸 𒅆 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒌉 𒂍 𒃻 𒁹𒀭𒂗𒆤𒈬𒈬
𒁹𒀭𒈦𒋀𒌶 𒇽𒀴 𒃻 𒁹𒀭𒂗𒀉𒆗𒉡
𒀸 𒋗𒈫


## 3. Get flat DataFrames (for analysis & export)

The function `get_full_flat_table` combines the processed content of the documents with the simplified metadata.

In [6]:
from oracc_parser import get_full_flat_table

# Full flat table — everything in one DataFrame
flat = get_full_flat_table(records)
print(f"Full table: {flat.shape[0]} rows × {flat.shape[1]} columns")
print(f"Columns: {list(flat.columns)}")
flat

Full table: 10 rows × 22 columns
Columns: ['id', 'project', 'text_id', 'genre', 'archive', 'provenance', 'pleiades_id', 'state_supergroup', 'period', 'start_year', 'end_year', 'accession_museum_publication_numbers', 'secondary_literature', 'credits', 'cite_as', 'transliteration', 'normalization', 'lemmatization', 'unicode', 'translation', 'total_tokens', 'tokens_without_broken']


,id,project,text_id,genre,archive,provenance,pleiades_id,state_supergroup,period,start_year,...,secondary_literature,credits,cite_as,transliteration,normalization,lemmatization,unicode,translation,total_tokens,tokens_without_broken
0,babcity_P261352,babcity,P261352,Legal,Murašû Archive,Nippur,912910,Mix,achaemenid,-547,...,Clay Aram.Endors. 306 #17(Aram.only); PBS 02/1...,Based on a transliteration by Heather D. Baker...,Please cite this page as http://oracc.org/babc...,1 1/2 MA.NA KU₃.BABBAR i-di E₂ ša₂ MU 1-KAM]\n...,UNK UNK manû kaspi idī bīti ša šatti UNK\nDari...,UNK UNK manû kaspu idu bītu ša šattu UNK\nDari...,𒁹 𒈦 𒈠𒈾 𒆬𒌓 𒄿𒁲 𒂍 𒃻 𒈬 𒁹𒄰\n𒁹𒁕𒊑𒅀𒀀𒈲 𒈗 𒃻 𒁹𒀭𒂗𒀉𒆗𒉡\n𒀀 𒃻 ...,,88,88
1,babcity_P285916,babcity,P285916,Legal,Egibi Archive,Babylon,893951,Mix,achaemenid,-547,...,"Zawadzki, Rental of Houses 130","Based on the edition of Stefan Zawadzki, The R...",Please cite this page as http://oracc.org/babc...,E₂ ša₂ {m}ni-din-ti-{d}EN DUMU {m}ši-rik\nša₂ ...,bītu ša Nidinti-Bel māri Širku\nša ṭāh bīt Suq...,bītu ša Nidinti-Bel māru Širku\nša ṭāh bītu Su...,𒂍 𒃻 𒁹𒉌𒁷𒋾𒀭𒂗 𒌉 𒁹𒅆𒋆\n𒃻 𒁕 𒂍 𒁹𒋻𒀀𒀀\n𒁹𒉌𒁷𒌈 𒌉 𒁹𒅆𒋆 𒀀 𒁹𒂊𒐕...,,92,92
2,babcity_P295135,babcity,P295135,Legal,Ea-ilutu-bani Archive,Borsippa,893964,Mix,achaemenid,-547,...,"Joannes,Archives de Borsippa 94 (Tr) and 325 (T)","Adapted, lemmatised and translated by Lindsey ...",Please cite this page as http://oracc.org/babc...,E₂-⸢su⸣ ša₂ {e₂}a-su-up ša₂ {m}⸢x⸣-[...]\nA-šu...,bīssu ša asup ša X\nmārišu ša Luṣi-ana-nur-Mar...,bītu ša asuppu ša X\nmāru ša Luṣi-ana-nur-Mard...,𒂍𒋢 𒃻 𒂍𒀀𒋢𒌒 𒃻 𒁹Xx\n𒀀𒋙 𒃻 𒁹𒇻𒍮𒀀𒈾𒂟𒀭𒀫𒌓 x\n𒀀𒈾 𒈬𒀭𒈾 𒈫 𒂆 ...,,84,81
3,babcity_P295289,babcity,P295289,Legal,Ea-ilutu-bani Archive,Borsippa,893964,Mix,Neo-Babylonian,-626,...,"Joannes, Archives de Borsippa 113 (Tr) and 323(T)","Adapted, lemmatised and translated by Lindsey ...",Please cite this page as http://oracc.org/babc...,6 GI-MEŠ KI{ti₃} E₂ {d}nam-tar-ri\nša₂ qe₂-reb...,UNK qanû erṣet bīt Namtar\nša qereb Borsippa š...,UNK qanû erṣetu bītu Namtar\nša qerbu Borsippa...,𒐋 𒄀𒈨𒌍 𒆠𒁴 𒂍 𒀭𒉆𒋻𒊑\n𒃻 𒆠𒆗 𒁈𒉺𒇻𒆠 𒃻 𒁹𒈾𒁲𒉡\n𒀀𒋙 𒃻 𒁹𒇻𒍪𒀀𒈾𒂟...,,89,89
4,babcity_P296350,babcity,P296350,Legal,Ea-ilutu-bani Archive,Borsippa,893964,Mix,Neo-Babylonian,-626,...,,"Adapted, lemmatised and translated by Lindsey ...",Please cite this page as http://oracc.org/babc...,E₂ ša₂ {f}GEME₂-{d}su-ti-ti DUMU.MI₃-su\nša₂ {...,bītu ša Amat-Sutiti mārassu\nša Iddinaya mār R...,bītu ša Amat-Sutiti mārtu\nša Iddinaya māru Ra...,𒂍 𒃻 𒊩𒊩𒆳𒀭𒋢𒋾𒋾 𒌉𒈨𒋢\n𒃻 𒁹𒋧𒈾𒀀 𒀀 𒇽𒃲𒀀𒋛𒂊\n𒀀𒈾 𒄿𒁲 𒂍 𒀸 𒅆 𒁹...,,96,95
5,babcity_P296437,babcity,P296437,Legal,Egibi Archive,Babylon,893951,Mix,achaemenid,-547,...,,Based on a transliteration by Heather D. Baker...,Please cite this page as http://oracc.org/babc...,E₂ ša₂ DA E₂ {m}{d}EN-TIN{iṭ} u₃ DA E₂\n{m}{d}...,bīt ša ṭāh bīt Bel-uballiṭ u ṭāh bīt\nMarduk-n...,bītu ša ṭāh bītu Bel-uballiṭ u ṭāh bītu\nMardu...,𒂍 𒃻 𒁕 𒂍 𒁹𒀭𒂗𒁷𒀉 𒅇 𒁕 𒂍\n𒁹𒀭𒀫𒌓𒈾𒈲𒌉𒍑 𒃻 𒁹𒀭𒀫𒌓𒉽𒀀 𒅇\n𒋀𒈨𒌍𒋙...,,110,110
6,babcity_P297595,babcity,P297595,Legal,Mušēzib-Marduk Archive,Uruk,912986,Mix,Neo-Assyrian,-911,...,"Beaulieu, CBCY 1, p. 29","Based on the edition of Grant Frame, The Archi...",Please cite this page as http://oracc.org/babc...,⸢ki⸣-i a-di lib₃-bi {iti}ŠU 4 1/2 MA.NA KU₃.BA...,kī adi libbi Duʾuzu UNK UNK manû kaspi\nrašûtu...,kī adi libbu Duʾuzu UNK UNK manû kaspu\nrašûtu...,𒆠𒄿 𒀀𒁲 𒊮𒁉 𒌗𒋗 𒐉 𒈦 𒈠𒈾 𒆬𒌓\n𒊏𒋗𒌅 𒃻 𒌋𒅗 𒁹𒀭𒂗𒋧𒈾 𒁹𒌇𒅆𒀭 𒌉𒋙 ...,,89,89
7,babcity_P299751,babcity,P299751,Legal,Mušēzib-Marduk Archive,Uruk,912986,Mix,Neo-Assyrian,-911,...,"Brinkman and Kennedy 1983: 40, L.4; Ellis 1984...","Based on the edition of Grant Frame, The Archi...",Please cite this page as http://oracc.org/babc...,⸢ṭup⸣-pi A.ŠA₃ {giš}SAR GIŠIMMAR zaq-pi u ki-š...,ṭuppi eqli kirî gišimmaru zaqpi u kišubbâ\nerṣ...,ṭuppu eqlu kirû gišimmaru zaqpu u kišubbû\nerṣ...,𒁾𒉿 𒀀𒊮 𒄑𒊬 𒊷 𒍠𒉿 𒌋 𒆠𒊒𒁀𒀀\n𒆠𒁴 𒀀𒇉 𒅖𒊺𒋾 𒀀𒃼 𒂍 𒀕𒆠\n𒍑 𒀭𒋫 ...,,161,156
8,babcity_P299753,babcity,P299753,Legal,Mušēzib-Marduk Archive,Uruk,912986,Mix,Neo-Assyrian,-911,...,"Brinkman and Kennedy 1983: 45, L.94; Ellis 198...","Based on the edition of Grant Frame, The Archi...",Please cite this page as http:

## 4. Export & see where output goes

The table above can be saved locally as a JSONL or CSV format. The output directory of this package is under DATA_DIR, called `output`. If you want to change it to another directory, change the value of the variable `out_dir`.

In [7]:
from oracc_parser.settings import OUTPUT_DIR

out_dir = OUTPUT_DIR # change value if you want to save the output somewhere else
out_dir.mkdir(parents=True, exist_ok=True)

jsonl_path = out_dir / f"{PROJECT.replace('/', '_')}.jsonl"
csv_path   = out_dir / f"{PROJECT.replace('/', '_')}.csv"

flat.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
flat.to_csv(csv_path, index=False)

print(f"Exported to:")
print(f"  JSONL: {jsonl_path}  ({jsonl_path.stat().st_size / 1024:.1f} KB)")
print(f"  CSV:   {csv_path}  ({csv_path.stat().st_size / 1024:.1f} KB)")
print(f"\nOutput directory: {out_dir}")

Exported to:
  JSONL: /content/drive/MyDrive/oracc_data/output/babcity.jsonl  (45.8 KB)
  CSV:   /content/drive/MyDrive/oracc_data/output/babcity.csv  (41.7 KB)

Output directory: /content/drive/MyDrive/oracc_data/output


## 5. Available projects and bulk download

Running the code below will display a table that lists every project that is available on Zenodo, how many texts it contains, and whether the word CSVs are already cached on local disk.


In [9]:
import pandas as pd
from oracc_parser.settings import CATALOGUE_DIR, WORD_CSV_DIR

rows = []
for cat_file in sorted(CATALOGUE_DIR.glob("*.csv")):
    slug = cat_file.stem
    project = slug.replace("-", "/", 1)
    umbrella = slug.split("-")[0]
    n_texts = sum(1 for _ in open(cat_file, encoding="utf-8")) - 1  # fast line count
    downloaded = (WORD_CSV_DIR / slug).exists() and any((WORD_CSV_DIR / slug).glob("*.csv"))
    rows.append({"project": project, "umbrella": umbrella, "texts": n_texts, "downloaded": "✓" if downloaded else ""})

available = pd.DataFrame(rows)
n_dl = (available["downloaded"] == "✓").sum()
print(f"{len(available)} projects, {available['texts'].sum()} texts total — {n_dl} already downloaded")
display(available)


138 projects, 55393 texts total — 1 already downloaded


,project,umbrella,texts,downloaded
0,adsd/adart1,adsd,417,
1,adsd/adart2,adsd,417,
2,adsd/adart3,adsd,417,
3,adsd/adart5,adsd,106,
4,adsd/adart6,adsd,180,
...,...,...,...,...
133,tcma/taban,tcma,14,
134,tcma/tsa1,tcma,17,
135,tcma/tsh1,tcma,262,
136,tcma/ugarit,tcma,5,


To download all the word-level CSVs for all projects at once, run the cell below. Once they are all avilable locally, you can export the entire dataset with the code from above.

Run time: 20-25 minutes

In [ ]:
# from collections import defaultdict
# from tqdm import tqdm
# from oracc_parser.download.fetch_data import extract_project_csvs

# # Group catalogue entries by umbrella
# umbrellas = defaultdict(list)
# for slug in sorted(f.stem for f in CATALOGUE_DIR.glob("*.csv")):
#     umbrellas[slug.split("-")[0]].append(slug)

# # Only download umbrella groups that are not fully on disk yet
# to_download = [
#     (umbrella, slugs) for umbrella, slugs in sorted(umbrellas.items())
#     if not all(
#         (WORD_CSV_DIR / s).exists() and any((WORD_CSV_DIR / s).glob("*.csv"))
#         for s in slugs
#     )
# ]

# if not to_download:
#     print("All projects already downloaded.")
# else:
#     print(f"Downloading {len(to_download)} umbrella group(s) covering "
#           f"{sum(len(s) for _, s in to_download)} project(s)...")
#     skipped = []
#     for umbrella, slugs in tqdm(to_download, unit="group"):
#         try:
#             extract_project_csvs(slugs[0].replace("-", "/", 1))
#         except (FileNotFoundError, ValueError) as e:
#             skipped.append((umbrella, str(e).split(":")[0]))
#     if skipped:
#         print(f"\nSkipped {len(skipped)} group(s) with no downloadable word CSVs:")
#         for name, reason in skipped:
#             print(f"  {name}: {reason}")
#     print("Done. Re-run the cell above to see updated download status.")

## What's next?

- **Explore what's in the dataset:** See notebook `02_reference_data.ipynb` for a full overview of collected projects, catalogues, provenance, periods, and other reference data.
- **Configure parsing:** See notebook `03_configure_and_export.ipynb` for masking PNs, dropping broken signs, and changing output formats.
- **understand the process:** see notebook `04_oracc_json_processing.ipynb` for in-depth explanations on how the package processes the original oracc files and metadata.